# ECON6083: Machine Learning in Economics
## Lecture 7 Exercise: DAGs, Bad Controls, and Sensitivity Analysis

**Coverage:** Lecture 7


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from scipy import stats

np.random.seed(42)



---

## Part 1: Bad Controls Simulation

In this part, you will simulate four classic DAG structures and observe how controlling for the wrong variable biases your estimate of the causal effect of $X$ on $Y$.

### Q1.1 Chain (Mediator Bias)

**Structure**: $X \to M \to Y$. The true total effect of $X$ on $Y$ is $2 \times 3 = 6$.

**Task**: Run two regressions:
1. $Y$ on $X$ only (correct total effect)
2. $Y$ on $X$ and $M$ (over-control)

Report and interpret the coefficients.

In [ ]:
n = 5000
X = np.random.normal(0, 1, n)
eps_M = np.random.normal(0, 1, n)
eps_Y = np.random.normal(0, 1, n)
M = 2 * X + eps_M
Y = 3 * M + eps_Y

# TODO: Regress Y on X only
____
print(f"Y ~ X:      coef = {model1.params[1]:.3f}  [True = 6.0]")

# TODO: Regress Y on X and M
____
print(f"Y ~ X + M:  coef = {model2.params[1]:.3f}  [Biased toward 0]")



### Q1.2 Collider (Selection Bias)

**Structure**: $X \to C \leftarrow Y$. Here $X$ and $Y$ are **independent** (true effect = 0).

**Task**: Run two regressions:
1. $Y$ on $X$ only
2. $Y$ on $X$ and $C$ (opens the collider path)

Report and interpret the coefficients.

In [ ]:
X = np.random.normal(0, 1, n)
Y = np.random.normal(0, 1, n)  # Independent of X!
eps_C = np.random.normal(0, 1, n)
C = X + Y + eps_C

model_noC = sm.OLS(Y, sm.add_constant(X)).fit()
# TODO: Regress Y on X and C
____
print(f"Y ~ X (no control):  coef = {model_noC.params[1]:.3f}  [True = 0.0]")
print(f"Y ~ X + C (collider): coef = {model_C.params[1]:.3f}  [Spurious!]")



### Q1.3 Descendant (Partial Collider Bias)

**Structure**: $X \to C \leftarrow Y$ and $D = C + \text{noise}$.

**Task**: Compare three regressions:
1. $Y$ on $X$ only
2. $Y$ on $X$ and $C$
3. $Y$ on $X$ and $D$

Explain why controlling for $D$ introduces partial bias.

In [ ]:
X = np.random.normal(0, 1, n)
Y = np.random.normal(0, 1, n)
eps_C = np.random.normal(0, 1, n)
eps_D = np.random.normal(0, 1, n)
C = X + Y + eps_C
D = C + eps_D

m1 = sm.OLS(Y, sm.add_constant(X)).fit()
m2 = sm.OLS(Y, sm.add_constant(np.column_stack([X, C]))).fit()
m3 = sm.OLS(Y, sm.add_constant(np.column_stack([X, D]))).fit()
print(f"Y ~ X (no control): coef = {m1.params[1]:.3f}  [True = 0.0]")
print(f"Y ~ X + C:          coef = {m2.params[1]:.3f}  [Full collider bias]")
print(f"Y ~ X + D:          coef = {m3.params[1]:.3f}  [Partial collider bias]")



### Q1.4 M-Bias

**Structure**: $U_1 \to X$, $U_1 \to Z \leftarrow U_2$, $U_2 \to Y$. True effect of $X$ on $Y$ = 2.

**Task**: Show that controlling for the pre-treatment variable $Z$ (the M-structure collider) **introduces** bias where none existed.

In [ ]:
n_mb = 2000
U1 = np.random.normal(0, 1, n_mb)
U2 = np.random.normal(0, 1, n_mb)
Z = 0.5 * U1 + 0.5 * U2 + np.random.normal(0, 0.5, n_mb)
X = 0.5 * U1 + np.random.normal(0, 0.5, n_mb)
Y = 2 * X + 0.5 * U2 + np.random.normal(0, 0.5, n_mb)

model_noZ = sm.OLS(Y, sm.add_constant(X)).fit()
# TODO: Regress Y on X and Z to demonstrate M-bias
____
print(f"Y ~ X (no Z):   coef = {model_noZ.params[1]:.3f}  [Correct, true = 2.0]")
print(f"Y ~ X + Z:      coef = {model_Z.params[1]:.3f}  [M-bias!]")



---

## Part 2: Sensitivity Analysis

You have estimated a treatment effect of $X$ on $Y$ using observed controls $W$. But what if there is an **unobserved confounder** $U$?

### Q2.1 Data Generation

Generate the data below and verify the DGP.

In [ ]:
n = 2000
U = np.random.normal(0, 1, n)
W = np.random.normal(0, 1, n)
eps_X = np.random.normal(0, 1, n)
eps_Y = np.random.normal(0, 1, n)
X = 0.5 * W + 0.3 * U + eps_X
Y = 2 * X + 0.5 * W + 0.4 * U + eps_Y



### Q2.2 Short vs Long Regression

Run three regressions using `statsmodels`:
1. **Short**: $Y$ on $X$ only
2. **Long**: $Y$ on $X$ and $W$ (observed control only)
3. **Oracle**: $Y$ on $X$, $W$, and $U$ (unrealistic, but gives the true effect)

Report the coefficient on $X$ and the $R^2$ for each regression.

In [ ]:
# TODO: Run short, long, and oracle regressions
short = ____  # TODO: Y on X only
long = ____   # TODO: Y on X and W
oracle = sm.OLS(Y, sm.add_constant(np.column_stack([X, W, U]))).fit()

print(f"Short:  coef = {short.params[1]:.3f}, R2 = {short.rsquared:.3f}")
print(f"Long:   coef = {long.params[1]:.3f}, R2 = {long.rsquared:.3f}")
print(f"Oracle: coef = {oracle.params[1]:.3f}, R2 = {oracle.rsquared:.3f}")



### Q2.3 Bias Decomposition

Compute the bias remaining after controlling for $W$, the bias eliminated by $W$, and the fraction of total bias eliminated.

In [ ]:
# TODO: Compute bias_remaining, bias_eliminated, and fraction_eliminated
bias_remaining = ____  # TODO: bias still left after controlling for W
bias_eliminated = short.params[1] - long.params[1]
total_bias = short.params[1] - oracle.params[1]
fraction_eliminated = bias_eliminated / total_bias if total_bias != 0 else np.nan

print(f"Bias remaining after W:       {bias_remaining:.3f}")
print(f"Bias eliminated by W:         {bias_eliminated:.3f}")
print(f"Fraction of total eliminated: {fraction_eliminated:.2%}")



### Q2.4 Sensemakr

Use the `sensemakr` package to quantify how strong an unobserved confounder would need to be to fully explain away the estimated treatment effect.

Install it with `!pip install pysensemakr` if needed.

In [ ]:
# If you haven't installed it yet, uncomment the next line:
# !pip install pysensemakr

import sensemakr

# We need to fit the long regression using a pandas DataFrame so sensemakr can parse variable names
df_sens = pd.DataFrame({'Y': Y, 'X': X, 'W': W})
long_sm = sm.OLS.from_formula('Y ~ X + W', data=df_sens).fit()

# TODO: Run sensemakr on the long regression with treatment='X'
sens = ____
print(sens.summary())

# TODO: What is the robustness value? What does it mean in one sentence?



### Q2.5 Sensitivity Contour Simulation

Suppose the unobserved confounder $U$ explains a fraction $r_Y$ of the residual variance in $Y$ and a fraction $r_X$ of the residual variance in $X$ (after partialling out $W$).

**Task**: For a grid of $r_X \in [0, 0.3]$ and $r_Y \in [0, 0.3]$, simulate the implied bias in $\hat{\beta}$ and draw a contour plot showing where the estimated effect would drop to specific values.

*Hint*: You can approximate the bias as $\text{bias} \approx \gamma_X \gamma_Y / \text{Var}(X_{resid})$, where $\gamma_X, \gamma_Y$ are the coefficients from regressing $U$ on residualized $X$ and $Y$.

In [ ]:
res_X_W = sm.OLS(X, sm.add_constant(W)).fit().resid
res_Y_W = sm.OLS(Y, sm.add_constant(W)).fit().resid

r_grid = np.linspace(0, 0.3, 30)
R_X, R_Y = np.meshgrid(r_grid, r_grid)

ratio_sd = np.std(res_Y_W) / np.std(res_X_W)
bias_grid = np.sqrt(R_X * R_Y) * ratio_sd

beta_long = long.params[1]
max_bias = np.max(bias_grid)
contour_levels = [beta_long - max_bias, beta_long - max_bias/2, beta_long]

plt.figure(figsize=(6, 5))
# TODO: Draw the contour plot of beta_long - bias_grid
contour = plt.contour(R_X, R_Y, beta_long - bias_grid, levels=contour_levels, colors=['red', 'orange', 'green'])
plt.clabel(contour, inline=True, fontsize=9)
plt.title('Sensitivity Contour: Unobserved Confounder Strength')
plt.xlabel('Partial R^2 of U on X (residualized)')
plt.ylabel('Partial R^2 of U on Y (residualized)')
plt.axvline(0, color='gray', linestyle='--', alpha=0.3)
plt.axhline(0, color='gray', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()



---

## Part 3: Causal Discovery

Can we learn the causal graph from data alone? In this part, you will implement a **very simplified** version of the PC algorithm skeleton phase.

### Q3.1 Generate a Faithful DAG

Generate data from the following DAG (all edges have coefficient 0.5):
- $A \to B$
- $A \to C$
- $B \to D$
- $C \to D$
- $B$ and $C$ are **not** directly connected.

So the true skeleton edges are: $(A,B), (A,C), (B,D), (C,D)$.

In [ ]:
n = 1000
A = np.random.normal(0, 1, n)
B = 0.5 * A + np.random.normal(0, 1, n)
C = 0.5 * A + np.random.normal(0, 1, n)
D = 0.5 * B + 0.5 * C + np.random.normal(0, 1, n)

data = pd.DataFrame({'A': A, 'B': B, 'C': C, 'D': D})
print(data.corr().round(3))



### Q3.2 Partial Correlation

Implement a helper that returns the partial correlation of $x$ and $y$ after conditioning on $z$, along with the two-sided p-value.

In [ ]:
def partial_correlation(x, y, z):
    """
    Return partial correlation of x and y after conditioning on z,
    and the two-sided p-value.
    """
    # TODO: Implement using OLS residuals
    # Step 1: regress x on z, get residuals x_res
    x_res = ____
    # Step 2: regress y on z, get residuals y_res
    y_res = ____
    # Step 3: compute Pearson correlation between x_res and y_res
    r, p = stats.pearsonr(x_res, y_res)
    return r, p

# Test your function
r1, p1 = partial_correlation(B, C, A)
print(f"B indep C | A:  partial r = {r1:.3f}, p = {p1:.3f}")

r2, p2 = stats.pearsonr(B, C)
print(f"B indep C:       r = {r2:.3f}, p = {p2:.3f}")

r3, p3 = partial_correlation(B, D, C)
print(f"B indep D | C:  partial r = {r3:.3f}, p = {p3:.3f}")



### Q3.3 Skeleton Discovery (PC Algorithm Phase I)

Using your `partial_correlation` function, implement the skeleton-discovery phase:
1. Start with all pairs. Keep an edge if the unconditional correlation is significant ($p < 0.05$).
2. For each remaining edge, test conditional independence given every other single variable.
3. Remove the edge if any conditioning set makes it independent ($p > 0.05$).

Compare your recovered skeleton to the true skeleton: `[('A','B'), ('A','C'), ('B','D'), ('C','D')]`

In [ ]:
variables = ['A', 'B', 'C', 'D']
alpha = 0.05

# Step 1: unconditional tests
skeleton = {}
for i, vi in enumerate(variables):
    for vj in variables[i + 1:]:
        # TODO: test unconditional correlation
        r, p = stats.pearsonr(data[vi], data[vj])
        if p < alpha:
            skeleton[(vi, vj)] = None

print("After unconditional tests, edges:", list(skeleton.keys()))

# Step 2: conditional tests for remaining edges
edges_to_remove = []
for (vi, vj) in list(skeleton.keys()):
    others = [v for v in variables if v != vi and v != vj]
    for vk in others:
        # TODO: test conditional independence given vk
        r, p = ____
        if p > alpha:
            edges_to_remove.append((vi, vj))
            print(f"  Remove {vi}-{vj} given {vk} (p={p:.3f})")
            break

for e in edges_to_remove:
    if e in skeleton:
        del skeleton[e]

print("Recovered skeleton:", list(skeleton.keys()))
print("True skeleton:     [('A', 'B'), ('A', 'C'), ('B', 'D'), ('C', 'D')]")



### Q3.4 Hidden Confounder

Now introduce a hidden variable $H$ that affects both $B$ and $C$:
- $B = 0.5 A + 0.5 H + \varepsilon_B$
- $C = 0.5 A + 0.5 H + \varepsilon_C$
- $D = 0.5 B + 0.5 C + \varepsilon_D$

Re-run the skeleton discovery algorithm from Q3.3 on this new data. What edges are incorrectly kept or removed? Why?

In [ ]:
# TODO: Generate data with hidden confounder H and re-run skeleton discovery
H = np.random.normal(0, 1, n)
B_h = 0.5 * A + 0.5 * H + np.random.normal(0, 1, n)
C_h = 0.5 * A + 0.5 * H + np.random.normal(0, 1, n)
D_h = 0.5 * B_h + 0.5 * C_h + np.random.normal(0, 1, n)

data_h = pd.DataFrame({'A': A, 'B': B_h, 'C': C_h, 'D': D_h})

# Copy your Q3.3 logic here and apply it to data_h
skeleton_h = {}
for i, vi in enumerate(variables):
    for vj in variables[i + 1:]:
        r, p = stats.pearsonr(data_h[vi], data_h[vj])
        if p < alpha:
            skeleton_h[(vi, vj)] = None

edges_to_remove_h = []
for (vi, vj) in list(skeleton_h.keys()):
    others = [v for v in variables if v != vi and v != vj]
    for vk in others:
        r, p = partial_correlation(data_h[vi], data_h[vj], data_h[vk])
        if p > alpha:
            edges_to_remove_h.append((vi, vj))
            print(f"  Remove {vi}-{vj} given {vk} (p={p:.3f})")
            break

for e in edges_to_remove_h:
    if e in skeleton_h:
        del skeleton_h[e]

print(f"Recovered skeleton (hidden H): {list(skeleton_h.keys())}")



*(Write your answer here)*

### Q3.5 Failure Mode I: Near-Unfaithfulness

In the real world, causal effects can nearly cancel out, making variables look independent even when they are connected in the DAG. This violates the **faithfulness** assumption.

**Task**: Generate data from the following DAG:
- $A \to B$ (coefficient = 0.6)
- $B \to C$ (coefficient = 0.5)
- $A \to C$ (coefficient = -0.3)

The total effect of $A$ on $C$ is approximately $0.6 \times 0.5 - 0.3 = 0$. Test the unconditional correlation between $A$ and $C$. What is the p-value? Would the PC algorithm's first step (unconditional independence testing) keep or remove the $A-C$ edge? What does this imply about the danger of relying on causal discovery without theory?

In [ ]:
n_faith = 1000
A_f = np.random.normal(0, 1, n_faith)
B_f = 0.6 * A_f + np.random.normal(0, 1, n_faith)
C_f = 0.5 * B_f - 0.3 * A_f + np.random.normal(0, 1, n_faith)

# TODO: Compute Pearson correlation between A and C, report r and p-value
r, p = stats.pearsonr(A_f, C_f)
print(f"A vs C: r = {r:.3f}, p = {p:.3f}")



### Q3.6 Failure Mode II: Small Samples and Low Power

Conditional independence tests have limited statistical power in small samples. The PC algorithm may fail to remove spurious edges because the p-value does not cross the 0.05 threshold.

**Task**: Use the *faithful* DAG from Q3.1 ($A \to B, A \to C, B \to D, C \to D$), but with only $n=200$ observations. Re-run the skeleton discovery algorithm from Q3.3. Does it still recover the true skeleton? Which edges, if any, are incorrectly kept or removed?

In [ ]:
n_small = 200
A_s = np.random.normal(0, 1, n_small)
B_s = 0.5 * A_s + np.random.normal(0, 1, n_small)
C_s = 0.5 * A_s + np.random.normal(0, 1, n_small)
D_s = 0.5 * B_s + 0.5 * C_s + np.random.normal(0, 1, n_small)

data_s = pd.DataFrame({'A': A_s, 'B': B_s, 'C': C_s, 'D': D_s})

# TODO: Re-run skeleton discovery from Q3.3 on data_s
# Hint: copy your logic from Q3.3 and apply it to data_s
# Print the recovered skeleton and compare to the true skeleton: [('A','B'), ('A','C'), ('B','D'), ('C','D')]



**Question**: Why is causal discovery particularly dangerous in economics, where sample sizes are often small and the true DAG is complex with many potential confounders? Write your answer below.

*(Write your answer here)*

---

## Part 4: DML Cannot Fix Bad Controls

One of the central messages of this course is that **machine learning solves estimation (the curse of dimensionality), but DAGs solve identification (what to control for)**.

In this part, you will use `DoubleML` — the same package from Lecture 5 — to estimate the causal effect of education on wages under three different control strategies. You will see that DML produces tight, precise confidence intervals even when the underlying identification is wrong.

### Q4.1 Data Generation

We use a unified data-generating process based on the **ability bias** story:

- **Ability** ($A$) is an observed confounder (we include it strategically).
- **Education** ($D$) is the treatment variable.
- **Wage** ($Y$) is the outcome.

**Scenario 1 — Correct Controls (Confounder only)**:  
$$Y = 2.4 D + 1.5 A + \varepsilon_Y$$  
Total causal effect of $D$ on $Y$ = **2.4**.

**Scenario 2 — Over-Control (Mediator)**:  
$$M = 0.5 D + \varepsilon_M$$  
$$Y = 2.0 D + 1.5 A + 0.8 M + \varepsilon_Y$$  
Total effect is still 2.4, but controlling for $M$ blocks part of the causal path. The estimated "direct effect" should be ~2.0.

**Scenario 3 — Collider Bias**:  
$$Y = 2.4 D + 1.5 A + \varepsilon_Y$$  
$$C = D + Y + \varepsilon_C$$  
Controlling for $C$ opens a collider path and biases the estimate away from 2.4.

**Task**: Generate the three datasets below.

In [ ]:
n_dml = 5000
np.random.seed(42)

# Common components
A = np.random.normal(0, 1, n_dml)
D = 0.8 * A + np.random.normal(0, 0.5, n_dml)

# Scenario 1: Correct (control A only)
Y1 = 2.4 * D + 1.5 * A + np.random.normal(0, 1, n_dml)
X1 = pd.DataFrame({'A': A})

# Scenario 2: Over-control (mediator M)
M = 0.5 * D + np.random.normal(0, 0.5, n_dml)
Y2 = 2.0 * D + 1.5 * A + 0.8 * M + np.random.normal(0, 1, n_dml)
X2 = pd.DataFrame({'A': A, 'M': M})

# Scenario 3: Collider bias
Y3 = 2.4 * D + 1.5 * A + np.random.normal(0, 1, n_dml)
C = D + Y3 + np.random.normal(0, 0.5, n_dml)
X3 = pd.DataFrame({'A': A, 'C': C})



### Q4.2 Run DoubleML

Install and import `doubleml`. Use `RandomForestRegressor` as the nuisance learner (as in Lecture 5). Run `DoubleMLPLR` for each of the three scenarios with `n_folds=3`.

Report for each scenario:
1. The estimated coefficient on $D$
2. The standard error
3. The 95% confidence interval

In [ ]:
# If doubleml is not installed, uncomment the next line:
# !pip install doubleml

import doubleml as dml
from sklearn.ensemble import RandomForestRegressor

ml_l = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
ml_m = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)

def run_dml(y, d, x, scenario_name):
    """Helper to run DoubleMLPLR and print results."""
    data_obj = dml.DoubleMLData.from_arrays(x=x.values, y=y, d=d)
    dml_obj = dml.DoubleMLPLR(data_obj, ml_l, ml_m, n_folds=3)
    # TODO: Fit the DoubleML object
    ____
    coef = dml_obj.coef[0]
    se = dml_obj.se[0]
    ci_low, ci_high = dml_obj.confint().iloc[0]
    print(f"\n{scenario_name}")
    print(f"  Estimated effect: {coef:.3f}")
    print(f"  Std Error:        {se:.3f}")
    print(f"  95% CI:           [{ci_low:.3f}, {ci_high:.3f}]")
    return coef, se

run_dml(Y1, D, X1, "Scenario 1: Correct")
run_dml(Y2, D, X2, "Scenario 2: Mediator")
run_dml(Y3, D, X3, "Scenario 3: Collider")



### Q4.3 Interpretation

**Task**: Answer the following questions in the markdown cell below.

1. In Scenario 2 (Mediator), does DML estimate the total effect or the direct effect? Why?
2. In Scenario 3 (Collider), is the estimate biased even though we control for the confounder $A$? Why?
3. What does this exercise teach us about the limits of DoubleML? Can it fix identification problems caused by bad controls?

*(Write your answers here)*

---

## Submission Checklist

- [ ] All code cells run without error
- [ ] Coefficients are reported and interpreted
- [ ] Figures are displayed inline
- [ ] Written answers are complete and concise